<a href="https://colab.research.google.com/github/Chandan-Mahajan/Student-Performance-Analytics-Engine/blob/main/Student_Performance_Analytics_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Student Performance Analytics Engine**
**Analyze a 500 x 6 marks matrix: subject and student statistics, rankings and outlier detection.**

**:1. Imports, seed, generate a 500x6 marks matrix**:

In [ ]:
import numpy as np
np.random.seed(42)
marks = np.random.randint(0, 101, size=(500, 6))
subjects = np.array(["Maths", "Physics", "Chemistry", "English", "Computer Science", "Hindi"])
student_ids = np.arange(1, 501)

print("Marks matrix shape:", marks.shape)
print("500 students Marks:")
print("Maths,", "Physics,", "Chemistry,", "English,", "Biology,", "CS")

print(marks)

Marks matrix shape: (500, 6)
500 students Marks:
Maths, Physics, Chemistry, English, Biology, CS
[[ 51  92  14  71  60  20]
 [ 82  86  74  74  87  99]
 [ 23   2  21  52   1  87]
 ...
 [ 62  18  22  28  29  38]
 [ 24 100   7  57  15  71]
 [  0   4   2  64   3  98]]


**2. sum, mean, median, std, var with axis=0 and axis=1**

In [ ]:
import pandas as pd

subject_stats_df = pd.DataFrame({
    "Sum": np.sum(marks, axis=0),
    "Mean": np.mean(marks, axis=0),
    "Median": np.median(marks, axis=0),
    "Std": np.std(marks, axis=0),
    "Variance": np.var(marks, axis=0)
}, index=subjects)

subject_stats_df = subject_stats_df.round(2)

print("Subject Stats shape:", subject_stats_df.shape)
print()
print(subject_stats_df.to_string())
print()
print()

means = np.mean(marks, axis=0)
stds = np.std(marks, axis=0)

hardest_subject = subjects[np.argmin(means)]
easiest_subject = subjects[np.argmax(means)]
most_inconsistent = subjects[np.argmax(stds)]

print(f"Hardest subject (lowest mean): {hardest_subject}")
print(f"Easiest subject (highest mean): {easiest_subject}")
print(f"Most inconsistent subject (highest std): {most_inconsistent}")

Subject Stats shape: (6, 5)

                    Sum   Mean  Median    Std  Variance
Maths             25128  50.26    50.0  30.27    916.06
Physics           24250  48.50    46.0  29.35    861.21
Chemistry         24499  49.00    48.0  29.13    848.37
English           23965  47.93    48.0  29.69    881.77
Computer Science  24096  48.19    47.0  29.82    889.03
Hindi             25704  51.41    51.0  30.00    900.28


Hardest subject (lowest mean): English
Easiest subject (highest mean): Hindi
Most inconsistent subject (highest std): Maths


**3. Z-scores via broadcasting; flag outliers; rank students**


In [ ]:
means = np.mean(marks, axis=0)
stds = np.std(marks, axis=0)
z_scores = (marks - means) / stds

flat_indices = np.argsort(np.abs(z_scores), axis=None)[::-1]

print("Top 10 best Scores (Outliers):")
print()
for i in range(10):
    r, c = np.unravel_index(flat_indices[i], z_scores.shape)

    s_id = student_ids[r]
    subj_name = subjects[c]
    mark_val = marks[r, c]
    z_val = z_scores[r, c]

    z_type = "HIGH" if z_val > 0 else "LOW"
    print(f"    Student {s_id:3d}   {subj_name:9s}   Mark: {mark_val:3d}   z = {z_val:+.2f} ({z_type})")

print("\n")

total_marks = np.sum(marks, axis=1)
student_ranks = np.argsort(total_marks)[::-1]

top_10_indices = student_ranks[:10]
top_students_df = pd.DataFrame({
    "Rank": np.arange(1, 11),
    "Student": student_ids[top_10_indices],
    "Total": total_marks[top_10_indices],
    "Average": np.round(total_marks[top_10_indices] / 6, 2)
})
print("Top 10 students:")
print(top_students_df.to_string(index=False))


hardest_subj = subjects[np.argmin(means)]

total_marks = np.sum(marks, axis=1)
topper_idx = np.argmax(total_marks)
topper_id = student_ids[topper_idx]
topper_marks = total_marks[topper_idx]

outlier_count = np.sum(np.abs(z_scores) > 2.0)
print()
print()
print("SUMMARY")
print("--------")
print(f"Hardest subject : {hardest_subj}")
print(f"Topper          : Student {topper_id} with {topper_marks} marks")
print(f"Outliers found  : {outlier_count}")

Top 10 best Scores (Outliers):

    Student  38   Physics     Mark: 100   z = +1.75 (HIGH)
    Student  86   Physics     Mark: 100   z = +1.75 (HIGH)
    Student 265   Physics     Mark: 100   z = +1.75 (HIGH)
    Student 466   Physics     Mark: 100   z = +1.75 (HIGH)
    Student 499   Physics     Mark: 100   z = +1.75 (HIGH)
    Student 476   Physics     Mark: 100   z = +1.75 (HIGH)
    Student 130   Physics     Mark: 100   z = +1.75 (HIGH)
    Student 186   Physics     Mark: 100   z = +1.75 (HIGH)
    Student   8   English     Mark: 100   z = +1.75 (HIGH)
    Student 191   English     Mark: 100   z = +1.75 (HIGH)


Top 10 students:
 Rank  Student  Total  Average
    1      270    504    84.00
    2        2    502    83.67
    3      233    455    75.83
    4      149    446    74.33
    5      355    441    73.50
    6      359    441    73.50
    7      387    438    73.00
    8      257    437    72.83
    9      310    434    72.33
   10       65    430    71.67


SUMMARY
--------

**4. Add total with hstack; export CSV with files.download**


In [ ]:
from google.colab import files
total_marks = np.sum(marks, axis=1, keepdims=True)

marks_with_total = np.hstack((marks, total_marks))

columns_list = list(subjects) + ["Total"]
df_export = pd.DataFrame(marks_with_total, columns=columns_list)
df_export.insert(0, "Student_ID", student_ids)

filename = "student_performance_with_totals.csv"
df_export.to_csv(filename, index=False)

files.download(filename)
print(f"Successfully exported and downloaded {filename}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully exported and downloaded student_performance_with_totals.csv
